# Fase 1 · Adquisición, ETL y análisis exploratorio multivariado

**Proyecto:** Analítica de datos y aprendizaje automático para la predicción de la calidad del concentrado en un proceso minero industrial
**Autores:** Brayan Estevan Zapata Mesa · Andrea Yizeth Vasquez Moreno · Oscar David Jiménez Ruiz
**Especialización en Analítica de Datos – Corporación Unificada Nacional de Educación Superior (CUN)**

Este cuaderno desarrolla el **objetivo específico 1** (etapas 1 y 2 de la ruta CRISP-DM):

1. Adquisición de los datos y ficha del conjunto.
2. Diccionario de datos.
3. Proceso ETL con tabla de conteos antes y después.
4. Análisis exploratorio multivariado (distribuciones, tendencia y estacionalidad, correlaciones de Pearson y Spearman, VIF, reactivos frente a la calidad).
5. Análisis adicionales para nivel Excelente: comparación de dos tratamientos de atípicos y estabilidad de los hallazgos por mes.

**Criterios de reproducibilidad**
- Ninguna cifra se escribe a mano: todas salen del código y se guardan en `reports/`.
- Semilla fija (`SEED = 42`).
- Las transformaciones que aprenden de los datos (imputación, escalado) **no** se ajustan aquí; se aplicarán dentro de un `Pipeline` en la fase de modelado.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. La descarga desde OpenML tarda unos minutos.

## 0. Configuración del entorno

In [ ]:
import os, re, glob, json, platform, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.seasonal import seasonal_decompose

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})

# Estructura de carpetas del proyecto
RUTAS = {"raw": "data/raw", "processed": "data/processed",
         "fig": "reports/figures", "tab": "reports/tables", "rep": "reports"}
for p in RUTAS.values():
    os.makedirs(p, exist_ok=True)

HOY = dt.date.today().isoformat()
VERSIONES = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
             "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__,
             "seaborn": sns.__version__}
print("Fecha de ejecución:", HOY)
print(json.dumps(VERSIONES, indent=2))

def guardar_tabla(df, nombre, index=False):
    ruta = os.path.join(RUTAS["tab"], f"{nombre}.csv")
    df.to_csv(ruta, index=index, encoding="utf-8-sig")
    return ruta

def guardar_figura(fig, nombre):
    ruta = os.path.join(RUTAS["fig"], f"{nombre}.png")
    fig.savefig(ruta)
    return ruta

## 1. Adquisición de los datos

**Fuente:** *Quality Prediction in a Mining Process* (Magalhães Oliveira, 2017), Kaggle, licencia CC0 1.0. Réplica en OpenML (ID 43311).

El cuaderno busca primero un CSV en `data/raw/` (por ejemplo, el archivo descargado de Kaggle y subido a Colab). Si no lo encuentra, descarga el conjunto desde OpenML con `fetch_openml` y guarda una copia con la fecha de descarga en el nombre.

In [ ]:
# Metadatos de cada variable: (nombre original, nombre corto, descripción, unidad, rol en el proyecto)
# Las unidades se toman de la descripción de columnas de la fuente en Kaggle: verificarlas allí.
COLUMNAS = [
    ("date", "fecha", "Fecha y hora del registro", "—", "Índice temporal"),
    ("% Iron Feed", "fe_alim", "Contenido de hierro en la alimentación", "%", "Predictora (alimentación)"),
    ("% Silica Feed", "sio2_alim", "Contenido de sílice en la alimentación", "%", "Predictora (alimentación)"),
    ("Starch Flow", "almidon", "Flujo de almidón (depresor)", "m³/h", "Predictora (reactivo)"),
    ("Amina Flow", "amina", "Flujo de amina (colector)", "m³/h", "Predictora (reactivo)"),
    ("Ore Pulp Flow", "flujo_pulpa", "Flujo de pulpa alimentada", "t/h", "Predictora (proceso)"),
    ("Ore Pulp pH", "ph", "pH de la pulpa", "adimensional", "Predictora (proceso)"),
    ("Ore Pulp Density", "densidad", "Densidad de la pulpa", "kg/cm³", "Predictora (proceso)"),
] + [
    (f"Flotation Column 0{i} Air Flow", f"aire_c{i}", f"Flujo de aire en la columna de flotación {i}", "Nm³/h",
     "Predictora (proceso)") for i in range(1, 8)
] + [
    (f"Flotation Column 0{i} Level", f"nivel_c{i}", f"Nivel de espuma en la columna de flotación {i}", "mm",
     "Predictora (proceso)") for i in range(1, 8)
] + [
    ("% Iron Concentrate", "fe_conc", "Contenido de hierro en el concentrado (laboratorio)", "%",
     "Excluida (fuga de información)"),
    ("% Silica Concentrate", "sio2_conc", "Contenido de sílice en el concentrado (laboratorio)", "%",
     "Variable objetivo"),
]
META = pd.DataFrame(COLUMNAS, columns=["original", "corto", "descripcion", "unidad", "rol"]).set_index("corto")
TARGET = "sio2_conc"
LAB_VARS = ["fe_alim", "sio2_alim", "fe_conc", "sio2_conc"]          # análisis de laboratorio (horarios)
REACTIVOS = ["almidon", "amina"]
PROCESO = [c for c in META.index if c not in LAB_VARS + ["fecha"]]   # sensores (cada 20 s)
PREDICTORAS = [c for c in META.index if c not in ["fecha", "fe_conc", TARGET]]

def normalizar(nombre):
    # Quita espacios, %, guiones bajos y mayúsculas para reconocer la columna en Kaggle u OpenML
    return re.sub(r"[^a-z0-9]", "", str(nombre).lower())

MAPA = {normalizar(o): c for o, c in zip(META["original"], META.index)}

def cargar_datos():
    locales = sorted(glob.glob(os.path.join(RUTAS["raw"], "*.csv")))
    if locales:
        ruta = locales[-1]
        df = pd.read_csv(ruta)
        origen = f"archivo local {os.path.basename(ruta)}"
    else:
        from sklearn.datasets import fetch_openml
        bunch = fetch_openml(data_id=43311, as_frame=True, parser="auto")
        df = bunch.frame.copy()
        ruta = os.path.join(RUTAS["raw"], f"mining_flotation_openml43311_{HOY}.csv")
        df.to_csv(ruta, index=False)
        origen = f"OpenML ID 43311, descargado el {HOY}"
    return df, ruta, origen

raw, RUTA_RAW, ORIGEN = cargar_datos()
raw = raw.rename(columns=lambda c: MAPA.get(normalizar(c), c))
faltan = [c for c in META.index if c not in raw.columns]
assert not faltan, f"No se reconocieron estas columnas: {faltan}. Columnas recibidas: {list(raw.columns)}"
raw = raw[list(META.index)]
print("Origen:", ORIGEN)
print("Copia guardada en:", RUTA_RAW)
print("Dimensiones originales:", raw.shape)
raw.head()

## 2. Ficha del conjunto de datos

In [ ]:
# Conversión de tipos (se documenta en el ETL): la fuente de Kaggle usa coma decimal
def a_numero(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False).str.strip(), errors="coerce")

df = raw.copy()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
for c in META.index.drop("fecha"):
    df[c] = a_numero(df[c])

FICHA = pd.DataFrame([
    ("Nombre", "Quality Prediction in a Mining Process"),
    ("Fuente y autor", "Kaggle, Eduardo Magalhães Oliveira (2017); réplica en OpenML (ID 43311)"),
    ("Enlace", "https://www.kaggle.com/datasets/edumagalhaes/quality-prediction-in-a-mining-process"),
    ("Licencia", "CC0 1.0 Universal (dominio público)"),
    ("Origen de esta copia", ORIGEN),
    ("Fecha de descarga / ejecución", HOY),
    ("Periodo cubierto", f"{df['fecha'].min():%Y-%m-%d %H:%M} a {df['fecha'].max():%Y-%m-%d %H:%M}"),
    ("Número de registros", f"{len(df):,}".replace(",", ".")),
    ("Número de variables", str(df.shape[1])),
    ("Frecuencia", "Sensores de proceso cada 20 s; análisis de laboratorio cada hora"),
    ("Variable objetivo", "% Silica Concentrate (sio2_conc)"),
    ("Datos personales", "No contiene"),
], columns=["Campo", "Descripción"])
guardar_tabla(FICHA, "tabla4_ficha_dataset")
FICHA

## 3. Diccionario de datos

In [ ]:
DICC = pd.DataFrame({
    "Variable (original)": META["original"],
    "Nombre en el código": META.index,
    "Tipo": ["Fecha y hora" if c == "fecha" else "Numérica" for c in META.index],
    "Descripción": META["descripcion"],
    "Unidad": META["unidad"],
    "Mínimo": [df[c].min() if c != "fecha" else df[c].min().strftime("%Y-%m-%d") for c in META.index],
    "Máximo": [df[c].max() if c != "fecha" else df[c].max().strftime("%Y-%m-%d") for c in META.index],
    "% faltantes": (df.isna().mean() * 100).round(2).values,
    "Rol": META["rol"],
}).reset_index(drop=True)
for col in ["Mínimo", "Máximo"]:
    DICC[col] = [round(v, 3) if isinstance(v, (float, np.floating)) else v for v in DICC[col]]
guardar_tabla(DICC, "tabla5_diccionario_datos")
DICC

## 4. Proceso ETL

Cada paso registra cuántos registros entran y cuántos salen; la tabla final se usa como **Tabla 6** del documento.

In [ ]:
CONTEOS = []
def registrar(paso, antes, despues, obs):
    CONTEOS.append({"Paso": paso, "Registros antes": antes, "Registros después": despues, "Observación": obs})

# 4.1 Tipos de dato
fallos = {c: int(df[c].isna().sum() - raw[c].isna().sum()) for c in META.index}
fallos = {c: v for c, v in fallos.items() if v > 0}
registrar("Carga y conversión de tipos", len(raw), len(df),
          "Fecha a datetime y variables a numérico" + (f"; valores no convertibles: {fallos}" if fallos else "; sin errores de conversión"))
print("Valores que no se pudieron convertir:", fallos or "ninguno")

# 4.2 Duplicados exactos
n0 = len(df)
dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
registrar("Eliminación de duplicados exactos", n0, len(df), f"{dup} filas idénticas eliminadas")
print("Duplicados exactos eliminados:", dup)

# 4.3 Faltantes
falt = df.isna().sum()
print("Faltantes por variable:")
print(falt[falt > 0] if falt.sum() else "No hay valores faltantes")
n0 = len(df)
df = df.dropna(subset=["fecha"]).reset_index(drop=True)
registrar("Revisión de faltantes", n0, len(df),
          f"{int(falt.sum())} celdas faltantes; solo se eliminan filas sin fecha. La imputación se hará en el Pipeline de la fase de modelado")

In [ ]:
# 4.4 Frecuencias mixtas: la variable objetivo es horaria y se repite en los registros de 20 s.
# Algunas horas tienen valores interpolados (más de un valor de sílice dentro de la misma hora).
df["hora"] = df["fecha"].dt.floor("h")
valores_por_hora = df.groupby("hora")[TARGET].nunique()
horas_interp = valores_por_hora[valores_por_hora > 1].index
n0 = len(df)
df = df[~df["hora"].isin(horas_interp)].reset_index(drop=True)
registrar("Eliminación de horas con objetivo interpolado", n0, len(df),
          f"{len(horas_interp)} de {len(valores_por_hora)} horas tenían más de un valor de sílice")
print(f"Horas con valores interpolados: {len(horas_interp)} de {len(valores_por_hora)}")

# 4.5 Agregación horaria: media para sensores, primer valor para análisis de laboratorio
n0 = len(df)
agg = {c: "mean" for c in PROCESO}
agg.update({c: "first" for c in LAB_VARS})
horaria = df.groupby("hora").agg(agg)
horaria["n_registros"] = df.groupby("hora").size()
horaria = horaria[list(META.index.drop("fecha")) + ["n_registros"]]
registrar("Agregación horaria", n0, len(horaria),
          "Sensores: media por hora; laboratorio: valor de la hora. Unidad de análisis: la hora")
print("Registros por hora (resumen):")
print(horaria["n_registros"].describe().round(1))

# 4.6 Continuidad temporal
rango = pd.date_range(horaria.index.min(), horaria.index.max(), freq="h")
faltantes_h = rango.difference(horaria.index)
bloques = (pd.Series(rango.isin(horaria.index), index=rango) != pd.Series(rango.isin(horaria.index), index=rango).shift()).cumsum()
huecos = (pd.Series(~rango.isin(horaria.index), index=rango).groupby(bloques).agg(["sum", "size"]))
huecos = huecos[huecos["sum"] > 0]
print(f"Horas esperadas en el periodo: {len(rango)}; horas presentes: {len(horaria)}; horas ausentes: {len(faltantes_h)}")
print("Hueco más largo (horas):", int(huecos["size"].max()) if len(huecos) else 0)

In [ ]:
# 4.7 Atípicos con el criterio IQR (se marcan, no se eliminan)
def limites_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

filas = []
marcas = pd.DataFrame(index=horaria.index)
for c in PREDICTORAS + [TARGET]:
    lo, hi = limites_iqr(horaria[c])
    m = (horaria[c] < lo) | (horaria[c] > hi)
    marcas[c] = m
    filas.append({"Variable": c, "Límite inferior": round(lo, 3), "Límite superior": round(hi, 3),
                  "Atípicos (horas)": int(m.sum()), "% atípicos": round(100 * m.mean(), 2),
                  "Mínimo": round(horaria[c].min(), 3), "Máximo": round(horaria[c].max(), 3)})
ATIPICOS = pd.DataFrame(filas).sort_values("% atípicos", ascending=False)
horaria["atipico_alguna"] = marcas.any(axis=1)
registrar("Marcado de atípicos (IQR 1,5)", len(horaria), len(horaria),
          f"{int(horaria['atipico_alguna'].sum())} horas con al menos una variable atípica; se marcan y no se eliminan")
guardar_tabla(ATIPICOS, "tabla_atipicos_iqr")
ATIPICOS.head(10)

In [ ]:
# Tabla de conteos antes y después (Tabla 6 del documento)
TABLA_CONTEOS = pd.DataFrame(CONTEOS)
guardar_tabla(TABLA_CONTEOS, "tabla6_conteos_etl")

# Conjunto preparado para la fase de modelado
RUTA_PROC = os.path.join(RUTAS["processed"], "flotacion_horaria.csv")
horaria.to_csv(RUTA_PROC, encoding="utf-8")
print("Conjunto horario guardado en:", RUTA_PROC, "| dimensiones:", horaria.shape)
TABLA_CONTEOS

## 5. Análisis exploratorio multivariado

Cada figura se guarda en `reports/figures/`. En el documento, cada una lleva un título que se entienda solo y dos o tres frases de interpretación.

### 5.1 Distribución de la variable objetivo

In [ ]:
y = horaria[TARGET]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2, 1]})
sns.histplot(y, bins=40, kde=True, ax=ax[0], color="#2f6f8f")
ax[0].set(title="Distribución del % de sílice en el concentrado (horario)", xlabel="% SiO₂ en el concentrado", ylabel="Horas")
sns.boxplot(x=y, ax=ax[1], color="#9cc3d5")
ax[1].set(title="Diagrama de caja", xlabel="% SiO₂ en el concentrado")
guardar_figura(fig, "fig02_distribucion_objetivo"); plt.show()

RESUMEN_Y = y.describe().to_frame("valor").T
RESUMEN_Y["asimetria"] = y.skew(); RESUMEN_Y["curtosis"] = y.kurt()
guardar_tabla(RESUMEN_Y.round(3), "tabla_resumen_objetivo")
RESUMEN_Y.round(3)

### 5.2 Distribución de las variables principales

In [ ]:
principales = ["amina", "almidon", "ph", "densidad", "flujo_pulpa", "fe_alim", "sio2_alim", "fe_conc"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.ravel(), principales):
    sns.histplot(horaria[c], bins=35, ax=ax, color="#5b8a72")
    ax.set(title=f"{META.loc[c, 'descripcion']}", xlabel=f"{c} ({META.loc[c, 'unidad']})", ylabel="")
    ax.title.set_fontsize(9)
fig.suptitle("Distribución horaria de las variables principales", y=1.01)
fig.tight_layout(); guardar_figura(fig, "fig03_distribucion_principales"); plt.show()

### 5.3 Tendencia y estacionalidad

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
y_full = y.reindex(rango)   # las horas ausentes quedan como huecos en la gráfica
ax.plot(y_full.index, y_full, lw=0.5, color="#9cc3d5", label="Valor horario")
ax.plot(y_full.rolling(24, min_periods=12).mean(), lw=1.6, color="#1d4e66", label="Media móvil de 24 h")
ax.set(title="Evolución del % de sílice en el concentrado", xlabel="Fecha", ylabel="% SiO₂")
ax.legend(); guardar_figura(fig, "fig04_serie_objetivo"); plt.show()

# Descomposición sobre el tramo continuo más largo (periodo diario = 24 h)
presente = pd.Series(rango.isin(horaria.index), index=rango)
tramos, ini = [], None
for t, ok in presente.items():
    if ok and ini is None:
        ini = t
    if not ok and ini is not None:
        tramos.append((ini, t - pd.Timedelta(hours=1))); ini = None
if ini is not None:
    tramos.append((ini, presente.index[-1]))
inicio, fin = max(tramos, key=lambda tr: tr[1] - tr[0])
largo = int((fin - inicio) / pd.Timedelta(hours=1)) + 1
print(f"Tramo continuo más largo: {inicio} a {fin} ({largo} horas)")
if largo >= 48:
    dec = seasonal_decompose(y.loc[inicio:fin].asfreq("h"), model="additive", period=24)
    fig = dec.plot(); fig.set_size_inches(12, 7)
    fig.suptitle("Descomposición del % de sílice (tramo continuo más largo, periodo 24 h)", y=1.01)
    guardar_figura(fig, "fig05_descomposicion"); plt.show()
    print("Varianza explicada por la componente estacional diaria:",
          round(dec.seasonal.var() / dec.observed.var(), 4))
else:
    print("El tramo continuo es demasiado corto para descomponer.")

perfil = pd.DataFrame({"hora_dia": y.index.hour, "dia_semana": y.index.dayofweek, "y": y.values})
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
sns.pointplot(data=perfil, x="hora_dia", y="y", errorbar=("ci", 95), ax=ax[0], color="#1d4e66", markersize=3)
ax[0].set(title="% SiO₂ medio por hora del día (IC 95 %)", xlabel="Hora", ylabel="% SiO₂")
ax[0].tick_params(axis="x", labelsize=7)
sns.pointplot(data=perfil, x="dia_semana", y="y", errorbar=("ci", 95), ax=ax[1], color="#1d4e66")
ax[1].set(title="% SiO₂ medio por día de la semana (0 = lunes)", xlabel="Día", ylabel="")
fig.tight_layout(); guardar_figura(fig, "fig06_perfiles_estacionales"); plt.show()

### 5.4 Correlaciones de Pearson y Spearman

In [ ]:
vars_corr = PREDICTORAS + ["fe_conc", TARGET]
pearson = horaria[vars_corr].corr(method="pearson")
spearman = horaria[vars_corr].corr(method="spearman")
for nombre, M, fname in [("Pearson", pearson, "fig07_correlacion_pearson"),
                         ("Spearman", spearman, "fig08_correlacion_spearman")]:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(M, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, linewidths=0.3,
                cbar_kws={"shrink": 0.7, "label": f"Coeficiente de {nombre}"}, ax=ax)
    ax.set_title(f"Matriz de correlación de {nombre} (datos horarios)")
    guardar_figura(fig, fname); plt.show()

CORR_Y = pd.DataFrame({"Pearson": pearson[TARGET], "Spearman": spearman[TARGET]}).drop(TARGET)
CORR_Y = CORR_Y.reindex(CORR_Y["Spearman"].abs().sort_values(ascending=False).index).round(3)
guardar_tabla(CORR_Y, "tabla_correlacion_con_objetivo", index=True)
print("Correlación entre % Fe y % SiO₂ del concentrado (fuga de información):",
      round(pearson.loc["fe_conc", TARGET], 3))
CORR_Y

### 5.5 Multicolinealidad (VIF)

In [ ]:
X_vif = add_constant(horaria[PREDICTORAS].dropna())
VIF = pd.DataFrame({"Variable": PREDICTORAS,
                    "VIF": [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(PREDICTORAS))]})
VIF["Nivel"] = pd.cut(VIF["VIF"], [0, 5, 10, np.inf], labels=["Bajo (<5)", "Moderado (5-10)", "Alto (>10)"])
VIF = VIF.sort_values("VIF", ascending=False).round(2)
guardar_tabla(VIF, "tabla_vif")
print(VIF["Nivel"].value_counts().to_string())
VIF

### 5.6 Reactivos frente a la calidad del concentrado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, c in enumerate(REACTIVOS):
    hb = axes[0, j].hexbin(horaria[c], horaria[TARGET], gridsize=35, cmap="Blues", mincnt=1)
    axes[0, j].set(title=f"{META.loc[c, 'descripcion']} frente a % SiO₂", xlabel=f"{c} ({META.loc[c, 'unidad']})", ylabel="% SiO₂")
    fig.colorbar(hb, ax=axes[0, j], label="Horas")
    deciles = pd.qcut(horaria[c], 10, duplicates="drop")
    g = horaria.groupby(deciles, observed=True)[TARGET].agg(["mean", "std", "count"])
    g["ic95"] = 1.96 * g["std"] / np.sqrt(g["count"])
    centros = horaria.groupby(deciles, observed=True)[c].median()
    axes[1, j].errorbar(centros, g["mean"], yerr=g["ic95"], fmt="o-", color="#1d4e66", capsize=3)
    axes[1, j].set(title=f"% SiO₂ medio por decil de {c} (IC 95 %)", xlabel=f"{c} (mediana del decil)", ylabel="% SiO₂ medio")
fig.tight_layout(); guardar_figura(fig, "fig09_reactivos_vs_silice"); plt.show()

## 6. Análisis para nivel Excelente

### 6.1 ¿Los hallazgos se sostienen en el tiempo? (correlación por mes)

In [ ]:
top = CORR_Y.drop(index="fe_conc", errors="ignore").head(8).index.tolist()
por_mes = (horaria.groupby(horaria.index.to_period("M"))
           .apply(lambda g: g[top].corrwith(g[TARGET], method="spearman")))
por_mes.index = por_mes.index.astype(str)
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(por_mes.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={"label": "Spearman con % SiO₂"})
ax.set(title="Estabilidad mensual de la correlación de Spearman con el % de sílice", xlabel="Mes", ylabel="")
guardar_figura(fig, "fig10_estabilidad_mensual"); plt.show()

signo = np.sign(por_mes).nunique()
ESTABILIDAD = pd.DataFrame({"Spearman global": CORR_Y.loc[top, "Spearman"],
                            "Mínimo mensual": por_mes.min(), "Máximo mensual": por_mes.max(),
                            "Signo estable": signo.eq(1)}).round(3)
guardar_tabla(ESTABILIDAD, "tabla_estabilidad_mensual", index=True)
ESTABILIDAD

### 6.2 Comparación de dos tratamientos de atípicos

Se comparan dos alternativas sobre las predictoras horarias: **(A) recorte a los límites IQR** (*winsorización*) y **(B) reemplazo por la mediana**. Se mide cuánto cambia cada una la media, la desviación estándar y la correlación de Spearman con el objetivo. Es un análisis exploratorio: el tratamiento elegido se ajustará solo con datos de entrenamiento dentro del `Pipeline` de la fase de modelado.

In [ ]:
filas = []
for c in PREDICTORAS:
    s = horaria[c]; lo, hi = limites_iqr(s); m = (s < lo) | (s > hi)
    if m.sum() == 0:
        continue
    A = s.clip(lo, hi)
    B = s.mask(m, s.median())
    base_rho = s.corr(horaria[TARGET], method="spearman")
    for nombre, t in [("A: recorte IQR", A), ("B: mediana", B)]:
        filas.append({"Variable": c, "Tratamiento": nombre, "% atípicos": round(100 * m.mean(), 2),
                      "Cambio media (%)": 100 * abs(t.mean() - s.mean()) / abs(s.mean()),
                      "Cambio desv. est. (%)": 100 * abs(t.std() - s.std()) / s.std(),
                      "Cambio Spearman (abs.)": abs(t.corr(horaria[TARGET], method="spearman") - base_rho)})
COMP = pd.DataFrame(filas)
if len(COMP):
    RESUMEN_COMP = COMP.groupby("Tratamiento")[["Cambio media (%)", "Cambio desv. est. (%)", "Cambio Spearman (abs.)"]].mean().round(4)
    guardar_tabla(COMP.round(4), "tabla_comparacion_atipicos_detalle")
    guardar_tabla(RESUMEN_COMP, "tabla_comparacion_atipicos_resumen", index=True)
    menos = RESUMEN_COMP["Cambio desv. est. (%)"].idxmin()
    print("Tratamiento que altera menos la dispersión de los datos:", menos)
    display(RESUMEN_COMP)
else:
    print("No se encontraron atípicos en las predictoras.")

## 7. Resumen de cifras para el documento

Este resumen reúne las cifras que van en la sección 10 del documento. Se guarda en `reports/resumen_fase1.json`.

In [ ]:
RESUMEN = {
    "fecha_ejecucion": HOY, "origen": ORIGEN, "versiones": VERSIONES,
    "registros_originales": int(len(raw)), "variables": int(raw.shape[1]),
    "periodo": [str(df["fecha"].min()), str(df["fecha"].max())],
    "duplicados_eliminados": dup, "celdas_faltantes": int(falt.sum()),
    "horas_totales": int(len(valores_por_hora)), "horas_interpoladas": int(len(horas_interp)),
    "horas_finales": int(len(horaria)), "horas_ausentes_en_rango": int(len(faltantes_h)),
    "horas_con_atipicos": int(horaria["atipico_alguna"].sum()),
    "objetivo_media": round(float(y.mean()), 3), "objetivo_mediana": round(float(y.median()), 3),
    "objetivo_min": round(float(y.min()), 3), "objetivo_max": round(float(y.max()), 3),
    "corr_fe_conc_vs_objetivo": round(float(pearson.loc["fe_conc", TARGET]), 3),
    "top5_spearman": CORR_Y.drop(index="fe_conc", errors="ignore").head(5)["Spearman"].to_dict(),
    "vif_altos": VIF.loc[VIF["VIF"] > 10, "Variable"].tolist(),
}
with open(os.path.join(RUTAS["rep"], "resumen_fase1.json"), "w", encoding="utf-8") as f:
    json.dump(RESUMEN, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(RESUMEN, ensure_ascii=False, indent=2, default=str))

## 8. Archivos generados

- `data/raw/`: copia de los datos con la fecha de descarga.
- `data/processed/flotacion_horaria.csv`: conjunto horario para la fase de modelado.
- `reports/tables/`: ficha (Tabla 4), diccionario (Tabla 5), conteos del ETL (Tabla 6), VIF, correlaciones, atípicos y estabilidad.
- `reports/figures/`: figuras del EDA.
- `reports/resumen_fase1.json`: cifras clave.

Para descargar todo desde Colab, ejecute la celda siguiente.

In [ ]:
# Empaqueta los resultados en un .zip (en Colab se descarga automáticamente)
import shutil
zip_ruta = shutil.make_archive(f"resultados_fase1_{HOY}", "zip", root_dir=".", base_dir="reports")
print("Archivo generado:", zip_ruta)
try:
    from google.colab import files
    files.download(zip_ruta)
except ImportError:
    pass